# 08. 일반 도로 사진 받기 (학습용 추가 데이터)

**왜**: 정답(바깥 밝기)은 사진이 아니라 그 시각 · 자리의 천리안 일사량에서 나와요. 그래서 터널이 안 보이는 도로 사진도 "이런 하늘 · 그림자면 밝기가 이 정도"를 배우는 데 쓸 수 있어요.
11/5 시험: 학습 사진을 46 → 131장으로 늘리니 ML 위험도F1이 올랐어요 (LightGBM+ .71 → .77, 시험은 입구·출구 사진 고정).

**규칙**
- 학습에만 씀. **시험(채점)은 입구 · 출구가 보이는 터널 사진으로만**
- 사람 분류 필요 없음. 자동으로 거름
  - 터널 안 사진 → 터널에서 충분히 떨어진 곳만 (터널 가운데에서 길이/2 + 500 m 밖)
  - 밤 사진 → 07~18시만, 천리안 일사량 0.05 이하 제외
  - 360° 사진 제외
- 날씨 · 계절 · 시간대가 **고르게** 섞이도록 뽑음 (달 × 시간대 칸마다 같은 수 목표)
- 한 자리 · 한 날에서는 1장만 (거의 같은 사진 방지)

**순서**: 1 자리 고르기 → 2 사진 목록 받기 → 3 고르게 뽑기 → 4 사진 받기 → 5 천리안 정답 → 6 사진 숫자 → `data/road_photos_train.csv`
**준비**: `sunsight_data/.env`의 `MAPILLARY_TOKEN`, `KMA_APIHUB_KEY` (03 · 04에서 쓰던 키). 실행 위치: `sunsight_video`

In [ ]:
import os, math, time, json
from pathlib import Path
import numpy as np, pandas as pd, requests

DATA = Path('data'); SD = Path('..') / 'sunsight_data'
IMG = DATA / 'road_photos'; IMG.mkdir(parents=True, exist_ok=True)
RAW = DATA / 'road_mapillary_raw.jsonl'
CACHE = DATA / 'gk2a_photo'; CACHE.mkdir(parents=True, exist_ok=True)

def env(key):
    for line in (SD / '.env').read_text(encoding='utf-8-sig').splitlines():
        if line.strip().startswith(key + '='): return line.split('=', 1)[1].strip().strip('"').strip("'")
    return os.environ.get(key)
TOKEN, AUTH_KEY = env('MAPILLARY_TOKEN'), env('KMA_APIHUB_KEY')
assert TOKEN and AUTH_KEY, 'sunsight_data/.env 에 MAPILLARY_TOKEN, KMA_APIHUB_KEY 필요'

N_POINTS = 1500      # 사진 찾을 고속도로 자리 수
TARGET = 2000        # 최종으로 남길 사진 수 (천리안 요청 수 ≈ 이 정도)
SEED = 0
print('준비 완료')

## 1. 사진 찾을 자리 고르기
- 고속도로 위 자리 = 사고 표(05 결과)의 좌표 (전국 고속도로에 퍼져 있음). 사고 여부와는 상관없이 **"고속도로 위 한 점"**으로만 씀
- 터널 근처는 뺌: 각 터널 가운데에서 (길이 ÷ 2 + 500 m) 안이면 제외

In [ ]:
t = pd.read_csv(SD / 'data' / '3_결과' / 'train_table.csv', usecols=['lat', 'lon'], low_memory=False)
pts = t.dropna().round(3).drop_duplicates().reset_index(drop=True)          # 약 100 m 격자로 중복 제거
tun = pd.read_csv(SD / 'data' / '2_중간' / 'tunnel_expressway.csv', encoding='utf-8-sig')
tun['mlat'] = (tun.start_lat + tun.end_lat) / 2; tun['mlon'] = (tun.start_lon + tun.end_lon) / 2
tun['r'] = tun.length_m.fillna(500) / 2 + 500
lat0 = np.radians(36.5)
def far_from_tunnels(la, lo):
    d = 111000 * np.hypot(tun.mlat - la, (tun.mlon - lo) * np.cos(lat0))
    return bool((d > tun.r).all())
pts = pts[[far_from_tunnels(a, b) for a, b in zip(pts.lat, pts.lon)]]
pts = pts.sample(min(N_POINTS, len(pts)), random_state=SEED).reset_index(drop=True)
print('자리', len(pts))

## 2. 자리마다 주변 사진 목록 받기 (사진 파일 아님)
자리 주변 ±110 m 상자. 1,500곳이면 10~20분. 끊겨도 다시 실행하면 이어서 받아요.

In [ ]:
FIELDS = 'id,captured_at,computed_compass_angle,compass_angle,geometry,computed_geometry,is_pano,thumb_1024_url'
D = 0.001
done = set()
if RAW.exists():
    for line in RAW.open(encoding='utf-8'): done.add(json.loads(line)['pt'])
with RAW.open('a', encoding='utf-8') as f:
    for k, p in pts.iterrows():
        if k in done: continue
        dlon = D / math.cos(math.radians(p.lat))
        bbox = f'{p.lon-dlon},{p.lat-D},{p.lon+dlon},{p.lat+D}'
        data = None
        for attempt in range(3):
            try:
                r = requests.get('https://graph.mapillary.com/images',
                                 params={'access_token': TOKEN, 'bbox': bbox, 'fields': FIELDS, 'limit': 500}, timeout=30)
                r.raise_for_status(); data = r.json().get('data', []); break
            except Exception as e:
                time.sleep(2 ** attempt)
        if data is None: print(k, '실패'); continue
        f.write(json.dumps({'pt': int(k), 'images': data}, ensure_ascii=False) + '\n')
        if k % 200 == 0: print(k, '/', len(pts))
        time.sleep(0.05)
print('완료')

## 3. 고르게 뽑기
- 낮(07~18시) · 360° 아님 · 2016년 이후(천리안2A는 2019년 7월부터라, 그 전은 ASOS로 대신)
- 한 자리 × 한 날 = 1장
- **달(12) × 시간대(3: 아침 7~10 · 낮 11~14 · 오후 15~18)** 36칸에서 칸마다 같은 수를 목표로 뽑음 → 겨울 · 아침 · 흐린 날이 너무 적지 않게

In [ ]:
rows = []
for line in RAW.open(encoding='utf-8'):
    rec = json.loads(line)
    for im in rec['images']:
        g = (im.get('computed_geometry') or im.get('geometry') or {}).get('coordinates')
        if not g or im.get('is_pano') or not im.get('thumb_1024_url'): continue
        rows.append({'pt': rec['pt'], 'image_id': str(im['id']), 'lon': g[0], 'lat': g[1],
                     'captured_at': pd.to_datetime(im['captured_at'], unit='ms', utc=True).tz_convert('Asia/Seoul').tz_localize(None),
                     'heading': im.get('computed_compass_angle', im.get('compass_angle')), 'thumb': im['thumb_1024_url']})
img = pd.DataFrame(rows).drop_duplicates('image_id')
img['hour'] = img.captured_at.dt.hour; img['month'] = img.captured_at.dt.month; img['date'] = img.captured_at.dt.date
img = img[img.hour.between(7, 18) & (img.captured_at.dt.year >= 2019)]
img = img.sample(frac=1, random_state=SEED).drop_duplicates(['pt', 'date'])        # 한 자리 × 한 날 = 1장
img['slot'] = img.month.astype(str) + '_' + pd.cut(img.hour, [6, 10, 14, 18], labels=['아침', '낮', '오후']).astype(str)
per = math.ceil(TARGET / img.slot.nunique())
pick = img.groupby('slot').head(per)   # apply 안 씀: 새 pandas는 apply에서 묶는 칸(slot)을 빼 버림
print('후보', len(img), '→ 고름', len(pick))
print(pd.crosstab(pick.month, pick.slot.str.split('_').str[1]))

## 4. 사진 받기 (1024px 썸네일)

In [ ]:
for r in pick.itertuples():
    fp = IMG / f'{r.image_id}.jpg'
    if fp.exists(): continue
    try: fp.write_bytes(requests.get(r.thumb, timeout=30).content)
    except Exception as e: print(r.image_id, e)
pick = pick[[ (IMG / f'{i}.jpg').exists() for i in pick.image_id ]]
pick.drop(columns='thumb').to_csv(DATA / 'road_photos_meta.csv', index=False, encoding='utf-8-sig')
print('받은 사진', len(pick))

## 5. 천리안 정답 + ASOS 구름 (04와 같은 방식)
- 요청은 (위경도 소수 2자리 · 시각) 마다 1번. 이미 받은 파일은 건너뜀 (04 · 06과 같은 폴더 공유)
- 먼저 시험 1번 출력 → 숫자면 다음 셀

In [ ]:
BASE_URL = 'https://apihub.kma.go.kr/api/typ01/cgi-bin/url/nph_sun_sat_ana_txt'
ph = pick.copy()
ph['t0'] = ph.captured_at.dt.floor('h'); ph['lat_r'], ph['lon_r'] = ph.lat.round(2), ph.lon.round(2)
ph['file'] = [f'{a:.2f}_{b:.2f}_{t:%Y%m%d%H}.txt' for a, b, t in zip(ph.lat_r, ph.lon_r, ph.t0)]
utc = lambda x: (x - pd.Timedelta(hours=9)).strftime('%Y%m%d%H%M')
ph['tm1'] = [utc(x - pd.Timedelta(hours=1)) for x in ph.t0]; ph['tm2'] = [utc(x + pd.Timedelta(hours=2)) for x in ph.t0]
def fetch(r):
    params = {'int': '30', 'tm1': r.tm1, 'tm2': r.tm2, 'lat': f'{r.lat_r:.4f}', 'lon': f'{r.lon_r:.4f}', 'authKey': AUTH_KEY}
    res = requests.get(BASE_URL, params=params, timeout=60); res.raise_for_status(); return res.text
r = next(ph.itertuples()); print(r.file); print(fetch(r)[:500])
print('요청할 것', ph.file.nunique(), '번')

In [ ]:
fails = 0
for r in ph.drop_duplicates('file').itertuples():
    f = CACHE / r.file
    if f.exists() and f.stat().st_size > 0: continue
    try: txt = fetch(r)
    except Exception as e: txt = None
    if not txt or 'error' in txt[:200].lower() or '인증' in txt[:200]:
        fails += 1
        if fails >= 5: print('연속 실패 5번 → 멈춤 (한도면 내일 다시)'); break
        continue
    fails = 0; f.write_text(txt, encoding='utf-8'); time.sleep(0.2)
print('받은 파일', sum((CACHE / x).exists() for x in ph.file.unique()), '/', ph.file.nunique())

In [ ]:
def parse(txt):
    lines = [l for l in txt.splitlines() if l.strip().startswith('|')]
    head = [x.strip() for x in lines[0].strip('|').split('|')]
    out = []
    for l in lines[1:]:
        d = dict(zip(head, [x.strip() for x in l.strip('|').split('|')]))
        for k in head[4:]:
            out.append({'time_kst': pd.to_datetime(k, format='%Y%m%d%H%M') + pd.Timedelta(hours=9), 'gk2a_icsr': pd.to_numeric(d[k], errors='coerce')})
    return out
g = []
for f in ph.file.unique():
    p = CACHE / f
    if p.exists():
        for x in parse(p.read_text(encoding='utf-8')): x['file'] = f; g.append(x)
g = pd.DataFrame(g).dropna().drop_duplicates(['file', 'time_kst'])
ph['tm'] = ph.captured_at.dt.ceil('h')
ph = ph.merge(g, left_on=['file', 'tm'], right_on=['file', 'time_kst'], how='left').drop(columns='time_kst')

st = pd.read_csv(SD / 'data' / '1_원본' / 'asos_stations.csv', encoding='utf-8-sig').rename(columns={'지점': 'stn_id', '위도': 'slat', '경도': 'slon'})
aso = pd.read_csv(SD / 'data' / '1_원본' / 'asos_hourly_all.csv', usecols=['stn_id', 'tm', 'icsr', 'dc10Tca'], parse_dates=['tm'])
st = st[st.stn_id.isin(aso.stn_id.unique())].reset_index(drop=True)
def nearest(lat, lon):
    d = np.hypot(st.slat - lat, (st.slon - lon) * math.cos(math.radians(lat))); return st.stn_id[d.idxmin()]
ph['stn_id'] = [nearest(a, b) for a, b in zip(ph.lat, ph.lon)]
ph = ph.merge(aso.rename(columns={'icsr': 'asos_icsr', 'dc10Tca': 'cloud'}), on=['stn_id', 'tm'], how='left')
ph['E'] = ph.gk2a_icsr
ph = ph[ph.E > 0.05]                                   # 천리안 값 있고 낮인 사진만 (ASOS로 대신하지 않음: 정답 기준 통일)
print('정답 붙은 사진', len(ph), '| 구름 있음', ph.cloud.notna().sum())
print('구름 분포', ph.cloud.round().value_counts().sort_index().to_dict())

## 6. 사진 숫자 10개 (07과 같은 함수)
터널이 없으니 `portal_ratio`는 빈칸으로 둠 (LightGBM은 빈칸을 건너뜀, 규칙 · 선형은 학습 중간값으로 채움)

In [ ]:
import numpy as np, cv2

def photo_features(path, tunnel_mask=None, size=640):
    """사진 한 장 → 숫자 10개. 노출이 바뀌어도 덜 변하는 '비율·모양' 숫자 위주.
    tunnel_mask: 71786처럼 터널 영역 표시가 있으면 (True=터널 구멍), 없으면 가운데 어두운 덩어리로 추정."""
    img = cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_COLOR)   # 한글 경로 대응
    if img is None: return None
    h, w = img.shape[:2]; s = size / max(h, w)
    img = cv2.resize(img, (int(w * s), int(h * s)))
    h, w = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(float)
    H, S, V = [hsv[..., i].astype(float) for i in range(3)]
    sky, road = gray[: h // 3], gray[2 * h // 3:]
    sky_hsv = (H[: h // 3], S[: h // 3], V[: h // 3])
    blue = ((sky_hsv[0] >= 90) & (sky_hsv[0] <= 130) & (sky_hsv[1] >= 50) & (sky_hsv[2] >= 100)).mean()
    # 터널 구멍: 표시가 있으면 그대로, 없으면 가운데 영역에서 가장 어두운 15%
    if tunnel_mask is None:
        c = gray[h // 4: 3 * h // 4, w // 4: 3 * w // 4]
        thr = np.percentile(c, 15)
        tunnel_mask = np.zeros_like(gray, bool)
        tunnel_mask[h // 4: 3 * h // 4, w // 4: 3 * w // 4] = c <= thr
    else:
        tunnel_mask = cv2.resize(tunnel_mask.astype(np.uint8), (w, h)) > 0
    ring = cv2.dilate(tunnel_mask.astype(np.uint8), np.ones((41, 41), np.uint8)).astype(bool) & ~tunnel_mask
    portal = gray[tunnel_mask].mean() + 1 if tunnel_mask.any() else np.nan
    around = gray[ring].mean() + 1 if ring.any() else np.nan
    return {
        'sky_mean': sky.mean() / 255,                        # 하늘 밝기
        'blue_sky': blue,                                     # 파란 하늘 비율
        'white_ratio': (gray >= 250).mean(),                  # 하얗게 날아간 비율
        'shadow_contrast': (np.percentile(road, 90) - np.percentile(road, 10)) / 255,  # 그림자 대비
        'road_mean': road.mean() / 255,                       # 도로 밝기
        'sky_road_ratio': (sky.mean() + 1) / (road.mean() + 1),
        'portal_ratio': np.log10(around / portal) if portal == portal else np.nan,  # 입구/주변 밝기비(로그)
        'saturation': S.mean() / 255,                         # 색 진하기
        'spread': (np.percentile(gray, 95) - np.percentile(gray, 5)) / 255,         # 밝기 퍼짐
        'mean_all': gray.mean() / 255,                        # 평균 밝기(노출 영향 큼, 비교용)
    }
FEATURES = ['sky_mean', 'blue_sky', 'white_ratio', 'shadow_contrast', 'road_mean',
            'sky_road_ratio', 'portal_ratio', 'saturation', 'spread', 'mean_all']


In [ ]:
rows = []
for r in ph.itertuples():
    f = photo_features(IMG / f'{r.image_id}.jpg')
    if f: f['image_id'] = r.image_id; f['portal_ratio'] = np.nan; rows.append(f)
out = ph.merge(pd.DataFrame(rows), on='image_id')
out['scene'] = out.groupby(['pt', 'date']).ngroup() + 200000          # 터널 사진 장면 번호와 안 겹치게
keep = ['image_id', 'scene', 'captured_at', 'lat', 'lon', 'heading', 'month', 'gk2a_icsr', 'asos_icsr', 'cloud', 'E'] + FEATURES
out[keep].to_csv(DATA / 'road_photos_train.csv', index=False, encoding='utf-8-sig')
print('저장 data/road_photos_train.csv', len(out), '장')